# Nemotron: check the CPU export path before loading weights

**You are here → 06 — Nemotron with ExecuTorch.**

**Main workflow:** understand a local AI request → optimize Qwen → optimize Nemotron.
**Optional extensions:** model-specific NIM deployment; separate CPU portability branch.

| **Step** | **Notebook** | **Purpose** |
|---|---|---|
| 0. Understand the request | [00 — How a local AI request works](00_local_serving.ipynb) | Follow a question through schema context, model serving and generated SQL, then evaluate the results to establish a quality baseline. |
| 1. Qwen optimization | [01 — Qwen: caching and quantization](01_qwen_cache_quantization.ipynb) | Compare FP16/AWQ W4A16 and prefix caching, with cold and warm requests reported separately. |
| 2. Nemotron optimization | [02 — Nemotron optimization](02_nemotron_optimization.ipynb) | Test schema context, supported prefix caching, concurrent requests, scheduler capacity and output budgets. |
| 3. Qwen NIM (optional) | [03 — Qwen through NIM](03_qwen_nim.ipynb) | Deploy Qwen in a container, check the plan format, and compare schema context and concurrent requests. |
| 4. Nemotron NIM (optional) | [04 — Nemotron through NIM](04_nemotron_nim.ipynb) | Deploy the hybrid model through NIM and compare schema context and request concurrency. |
| Optional CPU portability | [05 — Qwen with ExecuTorch](05_edge_portability_executorch.ipynb) | Follow Qwen3-4B conversion and CPU export, then inspect a generated SQL answer and its timing. |
| **→ Optional compatibility study (current)** | [06 — Nemotron with ExecuTorch](06_nemotron_executorch.ipynb) | Check whether the installed exporter accepts Nemotron and identify what a working port still needs. |


## Can we use the same CPU deployment approach for Nemotron?

Can the **same Nemotron 4B hybrid checkpoint** used for clinical SQL in chapter 02
be exported to ExecuTorch and run locally on CPU, as Qwen3-4B did in chapter 05?
We want to try the direct SQL question from chapter 05, with the same short
schema and reference-result check. First, though, the exporter needs a model
definition that can represent Nemotron's hybrid architecture.

**The check stopped at export configuration:** the installed ExecuTorch 1.5.1
LLM recipe doesn't accept this Nemotron model class. We didn't load weights or
run native inference, so there are no SQL or performance results for this path.

Install the lightweight [notebook environment](../SETUP.md), then **Run All**.
The cells display the saved compatibility check without downloading weights.
The optional command later in the notebook lets you repeat the config-only check
in a separate edge environment. This is a software compatibility check on the
Apple arm64 host; no CPU or GPU inference workload was executed.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.nemotron_edge_notebook import recorded_probe
probe = recorded_probe(ROOT)
print(probe['label'])
display(JSON({k: probe[k] for k in ('status', 'source', 'source_sha256', 'model_id',
    'model_revision', 'architecture', 'model_type', 'hybrid_override_pattern', 'versions')}))


recorded local compatibility probe — not a native inference run


<IPython.core.display.JSON object>

## Why the Qwen conversion doesn't carry over automatically

Chapter 02 served this pinned checkpoint through vLLM; chapter 04 used NIM.
To use the CPU path from chapter 05, we'd need an ExecuTorch `.pte` program
that preserves this model's calculations. The source weights are BF16;
choosing and validating an edge precision recipe would come after establishing
a working model conversion.

Nemotron-H combines Mamba state-space blocks, attention and feed-forward layers.
The Mamba blocks update recurrent state as tokens are processed. A model adapter
must carry those operations and states into the exported program. Qwen3's
attention-only conversion doesn't supply that mapping.

The check fetched only `config.json` at the **same base revision as chapter 02**.
It attempted both `nemotron_h` and `nemotron_3_nano_4b` in the installed exporter
configuration. The next cell shows which model classes the exporter accepts
and the errors returned for the two Nemotron names. It also shows the intended
measurements as unavailable, since configuration was the only stage reached.


In [2]:
display(JSON({'installed_model_types': probe['installed_model_types'],
              'configuration_attempts': probe['attempts'],
              'exporter_config_source_sha256': probe['exporter_config_source_sha256']}))
rows = [('SQL correctness', 'unavailable — no generation'),
        ('TTFT', 'unavailable — no inference'),
        ('End-to-end request latency', 'unavailable — no request'),
        ('CPU memory / GPU memory', 'unavailable — no native run')]
display(Markdown('| Intended measurement | Result |\n|---|---|\n' +
                 '\n'.join(f'| {metric} | {value} |' for metric, value in rows)))
display(JSON({k: probe[k] for k in ('weights_downloaded', 'graph_capture_attempted',
    'pte_produced', 'native_inference_run', 'sql_correctness', 'ttft_seconds',
    'end_to_end_seconds', 'peak_memory_bytes')}))


<IPython.core.display.JSON object>

| Intended measurement | Result |
|---|---|
| SQL correctness | unavailable — no generation |
| TTFT | unavailable — no inference |
| End-to-end request latency | unavailable — no request |
| CPU memory / GPU memory | unavailable — no native run |

<IPython.core.display.JSON object>

## Optional: repeat the compatibility check

Use the separate environment described in [the edge guide](../tutorial/EXECUTORCH_MAC.md)
with [requirements-edge.txt](../requirements-edge.txt). The following cell **prints**
the command; it launches no process. Run it from the repository root using your edge
Python. It downloads only the pinned config; choose a new output filename for each
check so you can compare results after changing the exporter version. No model
repository Python code is executed.


In [3]:
import shlex
import sys
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
command = ['.venv-edge/bin/python', 'scripts/tutorial/probe_nemotron_edge.py',
           '--output', f'experiment-tracker/tutorial/edge/nemotron-compatibility-{stamp}.json']
print('From the repository root, using your edge Python:')
print(shlex.join(command))


From the repository root, using your edge Python:
.venv-edge/bin/python scripts/tutorial/probe_nemotron_edge.py --output experiment-tracker/tutorial/edge/nemotron-compatibility-20261109T215719_168909Z.json


## What would a working Nemotron CPU port need?

1. Obtain or implement a Nemotron-H adapter that preserves Mamba state updates,
   attention caches and tokenizer behavior for the exact checkpoint.
2. Check converted unquantized logits against Hugging Face before quantization.
3. Validate graph capture, delegated operators, fallback coverage, `.pte` loading
   and native token generation. A config entry alone does not satisfy this step.
4. Execute the existing clinical SQL smoke and compare result rows; then time TTFT
   and complete requests, with startup reported separately. On CPU report process
   RSS; GPU memory requires a GPU backend and a separate measurement.

For now, chapter 05 supplies a working Qwen CPU example. Use chapters 02 and 04
for Nemotron's vLLM and NIM experiments. This compatibility check identifies
the next development task for an ExecuTorch port; it doesn't establish that such
a port is impossible or provide timings to compare with those serving paths.

Sources: [pinned NVIDIA configuration](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16/blob/dfaf35de3e30f1867dd8dbc38a7fc9fb52d3914f/config.json),
[ExecuTorch model workflows](https://github.com/pytorch/executorch/blob/main/examples/README.md),
and the installed recipe inventory and errors saved above.

Return to [Qwen CPU portability](05_edge_portability_executorch.ipynb) or
[Nemotron serving optimization](02_nemotron_optimization.ipynb).
